# Run Analysis

**NOTE:** To run this notebook, the prerequisite data must be downloaded. See `prepare_data.ipynb`.

### Imports

In [ ]:
import pandas as pd
import shap
import numpy as np 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, roc_auc_score
)
from tqdm import tqdm

import os

### Paths

In [ ]:
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')

fig_dir = os.path.join(os.path.dirname(os.getcwd()), 'figures')

feature_data_dir = os.path.join(data_dir, 'cerf_gridded_siting_parameters')

### Settings

In [ ]:
scenario_list = ['rcp45cooler_ssp3',
                 'rcp45cooler_ssp5',
                 'rcp85cooler_ssp3',
                 'rcp85cooler_ssp5',
                 'rcp45hotter_ssp3',
                 'rcp45hotter_ssp5',
                 'rcp85hotter_ssp3',
                 'rcp85hotter_ssp5'
                ]

tech_list =[
    'gridcerf_gas_cc_no-ccs_recirculating',
    'gridcerf_gas_turbine_no-ccs_no-cooling',
    'gridcerf_solar_pv_centralized_no-cooling',
    'gridcerf_wind_onshore_hubheight100_no-cooling',
    'gridcerf_wind_onshore_hubheight120_no-cooling',
    'gridcerf_wind_onshore_hubheight140_no-cooling',
    'gridcerf_wind_onshore_hubheight80_no-cooling'
    ]

### Run Random Forest and SHAP Analysis

In [ ]:
%%time

abs_shap_importance_list = []
mean_shap_importance_list = []
scenario_array = []
tech_array = []
lmp_array = []
ic_array = []
mean_lmp_array = []
mean_ic_array = []
shap_value_dict = {}

class_report_df = pd.DataFrame()
for i, tech in enumerate(tqdm(tech_list)):
    for scenario in scenario_list:
        
        fn = os.path.join(feature_data_dir, f'feature_analysis_balanced_{tech}_{scenario}.csv')
        
        df_balanced = pd.read_csv(fn)
        
        feature_cols = ['interconnection_cost', 'lmp']

        X = df_balanced[feature_cols]
        y = df_balanced['sited']
        
        # Split data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # base random forest model
        model = RandomForestClassifier(random_state=42)
        
        # hyperparameter options
        param_grid = {
            'n_estimators': [200, 500, 1000], # number of trees
            'max_depth': [None, 5, 10], # tree depth
            'min_samples_leaf': [1, 2, 5], # min samples at a leaf
            'max_features': [1, 2]  # since there are only 2 features
        }

        n_folds = 10 if len(df_balanced) < 1000 else 5
        
        # GridSearchCV
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=n_folds,                 
            n_jobs=4,            
            verbose=0,          
            scoring='accuracy'    
        )
        
        # conduct a grid search of models based on the parameter range
        grid_search.fit(X_train, y_train)
        
        # collect the best model
        best_model = grid_search.best_estimator_

        # get the prediction from the test set
        y_pred = best_model.predict(X_test)

        # get a table of classification report values
        r = classification_report(y_test, y_pred, output_dict=True)
        class_df = pd.DataFrame(r).reset_index(names='test')
        class_df['technology'] = tech
        class_df['scenario'] = scenario
        class_report_df = pd.concat([class_report_df, class_df])
        
        # Use SHAP to explain the model
        explainer = shap.TreeExplainer(best_model)
        shap_explainer = explainer(X_test)

        # pickle the shap explainer object to plot beeswarm plots
        pkl_dir = os.path.join(data_dir,'shap_values')
        os.makedirs(pkl_dir, exist_ok=True)
        with open(os.path.join(pkl_dir, f'{tech}_{scenario}_shap_values.pkl'), 'wb') as f:
            pickle.dump(shap_explainer, f)
            
        # get the SHAP values from the explainer as a numpy ndarray
        shap_values = shap_explainer.values
        
        # Calculate aggregate values for feature importance
        shap_importance = pd.DataFrame({
                                        "feature": X.columns,
                                        'mean_shap': shap_values[:, :, 1].mean(axis=0), # mean value
                                        "mean_abs_shap_value": abs(shap_values[:,:,1]).mean(axis=0) # mean of absolute value
                                            }).sort_values(by="mean_abs_shap_value", ascending=False)
        # mean of absolute value
        abs_shap_importance = shap_importance[['feature', 'mean_abs_shap_value']].copy()
        abs_shap_importance = abs_shap_importance.rename(columns={'mean_abs_shap_value':f'{tech}'})

        lmp_array.append(abs_shap_importance[abs_shap_importance.feature == 'lmp'][tech].iloc[0])
        ic_array.append(abs_shap_importance[abs_shap_importance.feature == 'interconnection_cost'][tech].iloc[0])

        # mean shap
        mean_shap_importance = shap_importance[['feature', 'mean_shap']].copy()
        mean_shap_importance = mean_shap_importance.rename(columns={'mean_shap':f'{tech}'})
        
        mean_lmp_array.append(mean_shap_importance[mean_shap_importance.feature == 'lmp'][tech].iloc[0])
        mean_ic_array.append(mean_shap_importance[mean_shap_importance.feature == 'interconnection_cost'][tech].iloc[0])
    
        tech_array.append(tech)
        scenario_array.append(scenario)


# change column order
class_report_df = class_report_df[['technology', 'scenario', 'test', '0.0', '1.0', 'accuracy', 'macro avg', 'weighted avg']]

# combine into formatted dataframe of results
shap_data_dict = { 'tech': tech_array,'scenario': scenario_array, 
                   'abs_lmp': lmp_array, 'abs_ic':ic_array,
                 'mean_lmp':mean_lmp_array, 'mean_ic':mean_ic_array}
shap_df = pd.DataFrame(shap_data_dict)
shap_df = pd.melt(shap_df, id_vars=['tech', 'scenario'], value_vars=['abs_lmp', 'abs_ic'], 
                  var_name='feature', value_name='abs_shap')

In [ ]:
# save to file
shap_df.to_csv(os.path.join(data_dir, 'feature_analysis_output.csv'), index=False) 
class_report_df.to_csv(os.path.join(data_dir, 'feature_analysis_classification_report.csv'), index=False) 